# LSTM-Autoencoder Training on Google Colab (T4 GPU)
## Cold Chain EWS Digital Twin — Milestone 4a / 4b

---

> **⚠ STRUCTURAL PROXY NOTICE**  
> All data in this notebook derives from `IOT-temp.csv` — a **generic IoT temperature sensor log,  
> NOT real refrigerated-transport telemetry**. All features and models are **synthetic proxy constructs**.  
> No loss value or reconstruction threshold makes any food-safety claim.  
> See `docs/data_profile.md` and `docs/AD_LOG.md` (decisions D1, D9, D10, D11, D12).

---

### Student Instructions for Colab Run
1. **GPU Runtime**: Verify you are using a GPU runtime:  
   Navigate to **Runtime > Change runtime type > Hardware accelerator > T4 GPU**.
2. **Upload Data**: Create the folder `data/processed/` in Colab's file browser and upload the four window files produced by `src/lstm_prep.py`:
   - `lstm_windows_out_train.npz`
   - `lstm_windows_out_val.npz`
   - `lstm_windows_in_train.npz`
   - `lstm_windows_in_val.npz`
3. **Run All**: Execute the notebook from top to bottom (**Runtime > Run all**).
4. **Download Artifacts**: After training completes, download the generated files to your local repository:
   - `models/lstm_out.h5` → local `coldchain-ews-twin/models/lstm_out.h5`
   - `models/lstm_in.h5` → local `coldchain-ews-twin/models/lstm_in.h5`
   - `data/processed/lstm_training_log_out.json` → local `coldchain-ews-twin/data/processed/lstm_training_log_out.json`
   - `data/processed/lstm_training_log_in.json` → local `coldchain-ews-twin/data/processed/lstm_training_log_in.json`


In [ ]:
# 1. Hardware & GPU Environment Verification
import tensorflow as tf
import os

print("TensorFlow Version:", tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print("Physical GPUs Detected:", gpus)

# Enforce that training occurs on a GPU
assert len(gpus) > 0, "No GPU detected! Please navigate to Runtime > Change runtime type > T4 GPU."

# Display GPU hardware specifications via nvidia-smi
!nvidia-smi


In [ ]:
# 2. Directory Setup & Data Verification
import pathlib
import numpy as np

# Create local directories matching the repository structure
pathlib.Path("models").mkdir(parents=True, exist_ok=True)
pathlib.Path("data/processed").mkdir(parents=True, exist_ok=True)

# Verify presence of all required input files
required_files = [
    "data/processed/lstm_windows_out_train.npz",
    "data/processed/lstm_windows_out_val.npz",
    "data/processed/lstm_windows_in_train.npz",
    "data/processed/lstm_windows_in_val.npz",
]

for f in required_files:
    p = pathlib.Path(f)
    if not p.exists() and pathlib.Path(p.name).exists():
        pathlib.Path(p.name).rename(p)
    assert p.exists(), f"Missing required file: {f}. Please upload it before continuing."
    data = np.load(p)["windows"]
    print(f"Verified {f:42s}: shape={data.shape}, dtype={data.dtype}")


In [ ]:
# 3. Model Architecture Definition (D12)
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, RepeatVector, TimeDistributed, Dense
from tensorflow.keras.callbacks import EarlyStopping
import time
import json

LSTM_WINDOW_LENGTH = 30
N_FEATURES = 2
BATCH_SIZE = 256
MAX_EPOCHS = 50
PATIENCE = 5

def build_lstm_autoencoder(window_length=LSTM_WINDOW_LENGTH, n_features=N_FEATURES):
    """
    Builds the D12 LSTM-Autoencoder architecture:
    Encoder LSTM(32) -> Bottleneck -> RepeatVector(30) -> Decoder LSTM(32, return_sequences=True) -> TimeDistributed(Dense(2))
    Adam optimizer, MSE loss.
    """
    model = Sequential([
        LSTM(32, activation="tanh", input_shape=(window_length, n_features), name="encoder_lstm"),
        RepeatVector(window_length, name="bottleneck_repeat"),
        LSTM(32, activation="tanh", return_sequences=True, name="decoder_lstm"),
        TimeDistributed(Dense(n_features), name="reconstruction_dense")
    ])
    model.compile(optimizer="adam", loss="mse")
    return model

# Inspect model summary
sample_model = build_lstm_autoencoder()
sample_model.summary()


In [ ]:
# 4. Train Series Out LSTM-Autoencoder
X_out_train = np.load("data/processed/lstm_windows_out_train.npz")["windows"]
X_out_val   = np.load("data/processed/lstm_windows_out_val.npz")["windows"]

print(f"Training Series Out: Train shape={X_out_train.shape}, Val shape={X_out_val.shape}")

model_out = build_lstm_autoencoder()

early_stopping_out = EarlyStopping(
    monitor="val_loss",
    patience=PATIENCE,
    restore_best_weights=True,
    verbose=1
)

start_time_out = time.time()
history_out = model_out.fit(
    X_out_train, X_out_train,
    validation_data=(X_out_val, X_out_val),
    epochs=MAX_EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[early_stopping_out],
    verbose=1
)
elapsed_out = time.time() - start_time_out

# Save trained weights
model_out_path = "models/lstm_out.h5"
model_out.save(model_out_path)
print(f"Saved Series Out model to: {model_out_path}")

# Extract GPU information
gpu_device_name = tf.test.gpu_device_name()
try:
    gpu_hardware = tf.config.experimental.get_device_details(tf.config.list_physical_devices('GPU')[0]).get('device_name', gpu_device_name)
except Exception:
    gpu_hardware = gpu_device_name

log_out = {
    "series": "Out",
    "architecture": "LSTM(32)-RepeatVector(30)-LSTM(32)-Dense(2)",
    "n_train_windows": len(X_out_train),
    "n_val_windows": len(X_out_val),
    "batch_size": BATCH_SIZE,
    "max_epochs": MAX_EPOCHS,
    "epochs_trained": len(history_out.history["loss"]),
    "final_train_loss": float(history_out.history["loss"][-1]),
    "final_val_loss": float(history_out.history["val_loss"][-1]),
    "best_val_loss": float(min(history_out.history["val_loss"])),
    "training_wall_clock_sec": round(elapsed_out, 2),
    "gpu_device": str(gpu_hardware),
}

with open("data/processed/lstm_training_log_out.json", "w", encoding="utf-8") as f:
    json.dump(log_out, f, indent=2)
print("Saved training log: data/processed/lstm_training_log_out.json")


In [ ]:
# 5. Train Series In LSTM-Autoencoder
X_in_train = np.load("data/processed/lstm_windows_in_train.npz")["windows"]
X_in_val   = np.load("data/processed/lstm_windows_in_val.npz")["windows"]

print(f"Training Series In: Train shape={X_in_train.shape}, Val shape={X_in_val.shape}")

model_in = build_lstm_autoencoder()

early_stopping_in = EarlyStopping(
    monitor="val_loss",
    patience=PATIENCE,
    restore_best_weights=True,
    verbose=1
)

start_time_in = time.time()
history_in = model_in.fit(
    X_in_train, X_in_train,
    validation_data=(X_in_val, X_in_val),
    epochs=MAX_EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[early_stopping_in],
    verbose=1
)
elapsed_in = time.time() - start_time_in

# Save trained weights
model_in_path = "models/lstm_in.h5"
model_in.save(model_in_path)
print(f"Saved Series In model to: {model_in_path}")

log_in = {
    "series": "In",
    "architecture": "LSTM(32)-RepeatVector(30)-LSTM(32)-Dense(2)",
    "n_train_windows": len(X_in_train),
    "n_val_windows": len(X_in_val),
    "batch_size": BATCH_SIZE,
    "max_epochs": MAX_EPOCHS,
    "epochs_trained": len(history_in.history["loss"]),
    "final_train_loss": float(history_in.history["loss"][-1]),
    "final_val_loss": float(history_in.history["val_loss"][-1]),
    "best_val_loss": float(min(history_in.history["val_loss"])),
    "training_wall_clock_sec": round(elapsed_in, 2),
    "gpu_device": str(gpu_hardware),
}

with open("data/processed/lstm_training_log_in.json", "w", encoding="utf-8") as f:
    json.dump(log_in, f, indent=2)
print("Saved training log: data/processed/lstm_training_log_in.json")


In [ ]:
# 6. Loss Curves & Training Diagnostic
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

for ax, hist, s_name in zip(axes, [history_out, history_in], ["Out", "In"]):
    ax.plot(hist.history["loss"], label="Train Loss (MSE)", color="#2b8cbe", linewidth=2)
    ax.plot(hist.history["val_loss"], label="Val Loss (MSE)", color="#e41a1c", linewidth=2)
    ax.set_title(f"Reconstruction Loss: Series '{s_name}'", fontsize=12, fontweight="bold")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("MSE Loss")
    ax.legend(frameon=True)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Training Run Summary:")
print("Series Out:", log_out)
print("Series In :", log_in)


### 7. Download Checklist for Local Repository
Download the following 4 files from Colab to your local machine:
1. `models/lstm_out.h5` → place in `coldchain-ews-twin/models/lstm_out.h5`
2. `models/lstm_in.h5` → place in `coldchain-ews-twin/models/lstm_in.h5`
3. `data/processed/lstm_training_log_out.json` → place in `coldchain-ews-twin/data/processed/lstm_training_log_out.json`
4. `data/processed/lstm_training_log_in.json` → place in `coldchain-ews-twin/data/processed/lstm_training_log_in.json`

Once downloaded, Milestone 4b (test window extraction, reconstruction scoring, and lead-time evaluation) can proceed.
